In [1]:
print("please enter the buying price of the car: vhigh, high, med, or low")   
buying = input("buying: ")   

print("please enter the safety of the car: high, med, or low")   
safety = input("safety: ")   

print("please enter the maint price of the car: vhigh, high, med, or low")   
maint = input("maint: ")  
#print(terms)- allows me to check the user input converted correctly 

please enter the buying price of the car: vhigh, high, med, or low
please enter the safety of the car: high, med, or low
please enter the maint price of the car: vhigh, high, med, or low


In [2]:
row = {"buying": buying, 
        "maint":maint,
        "safety":safety}
    #{"buying:" "low", "safety:" "high", "maint:" "med"}

facts = {f"{attribute}={value}" for attribute, value in row.items()}#each attribute, plus its value from row become a fact

rules = [#  list of tuples 
#seeing that 70% of the dataset is unacc i made my default unacc, i focused on making rules for acc, good and , good, based of my research on the important factors of can determine the class of a car.
    ({("safety=low")}, ("class_value=unacc")),
    ({("maint=med"),("buying=low")}, ("class_value=good")),
    ({("maint=low"),("buying=low")}, ("class_value=good")),
    ({("safety=med"),("buying=high")}, ("class_value=acc")),
    ({("safety=med"),("buying=vhigh")}, ("class_value=acc")),
    ({("safety=med"),("buying=med")}, ("class_value=acc")),
    ({("safety=high"),("maint=med"),("buying=med")}, ("class_value=good")),
    ({("safety=high"),("maint=low")}, ("class_value=vgood")),
]



In [3]:
def backward_chain(goal, facts, rules):
    if goal in facts:
        return True#checks if the goal is already present in the known facts, if it is then true is returned 
    for conditions, conclusion in rules: #checks each rule with a matching goal. 
        if conclusion == goal:#if conditions match, then the conclusion is checked to see if the goal matches
            if all(backward_chain(c, facts, rules) for c in sorted(conditions)):#self calls to check each condition is true if so then goal is proved
                return True
    return False # if a goal cannot be proved then the outcome is false

In [4]:
#same function as backward_chain, but with the addition of an explanation so a user can see how the goal was proved.
    
def prove(goal, facts, rules, depth = 0):# depth keeps track of how many layers deeps the reasoning goes
    indent = " " * depth# for each layer of reasoning another indent is added
    if goal in facts:
        print(f"{indent}{goal}: known facts")#if goal is found, indents line, prints goal and "known facts" showing the goal is a known fact
        return True
    #not needed for the current rules i have but i decided to keep the code in case i update my rules in the future.
    for conditions, conclusion in rules:#checks each rule with a matching goal. 
        if conclusion == goal:#if conditions match, then the conclusion is checked to see if the goal matches
            print(f"{indent}{goal}: try rule needing {sorted(conditions)}")#prints the indent,goal and a rule with...conditions
            if all(prove(c, facts, rules, depth +1) for c in sorted(conditions)):#self calls to check each condition is true  if so then goal is proved
                print(f"{indent}{goal}: proved")# indents and prints the goal with "proved"
                return True
    print(f"{indent}{goal}: cannot be proved")
    return False


In [5]:
selected_goal = input("please type either unacc, acc, good, or vgood")

In [6]:
goal = f"class_value={selected_goal}"
print (goal)

class_value=acc


In [7]:
print(backward_chain(goal, facts, rules))#prints conclusion true or false

False


In [8]:
prove(goal, facts, rules)#shows the reasoning

class_value=acc: try rule needing ['buying=high', 'safety=med']
 buying=high: cannot be proved
class_value=acc: try rule needing ['buying=vhigh', 'safety=med']
 buying=vhigh: cannot be proved
class_value=acc: try rule needing ['buying=med', 'safety=med']
 buying=med: cannot be proved
class_value=acc: cannot be proved


False

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
attributes = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class_value"]# names each attribute in the dataset using a list.
carData = pd.read_csv("car.data", names=attributes, header = None )#reads the attributes in the dataset matching them up to the attribute names in the list
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
train, test = train_test_split(#train,
    carData, test_size = 0.3, random_state=42 # 30% of the data set to test on and randomly shuffles data
)

In [10]:
classes = ["unacc", "acc", "good", "vgood"]
def classify(r):#the connection to the backwards chaining function and the test data

   
    test_facts = {#testing data is stored in these rows
        f"buying={r['buying']}",
        f"maint={r['maint']}",
        f"safety={r['safety']}"

    }

   

    for class_value in classes:#checks all possible goals 
        goal =f"class_value= {class_value}"#passes each possible goal through the backward_chain function

        if backward_chain(goal, test_facts, rules):# if true adds the true goal to the list
            return class_value

    return "unacc"

In [11]:
# calculates the percentage of the most common class value, if i were to make everything unacc, then i would have a 70% accurate algorithm

# calculates the percentage of the most common class value, if i were to make everything unacc, then i would have a 69% accurate algorithm
#calculates the baseline accuracy and the accuracy of my rules.
from collections import Counter
y_true = test["class_value"].tolist()
majority = Counter(y_true).most_common(1)[0][0]

baseline = sum(value == majority for value in y_true) /len(y_true)

y_pred = [classify(row) for i, row in test.iterrows()]

accuracy = accuracy_score(y_true, y_pred)

print(f"always predicts {majority}: {baseline:.2f}")
print(f"my rules:{accuracy:.2f}")

always predicts unacc: 0.69
my rules:0.69


In [12]:

y_true = test["class_value"].tolist()
y_pred = [classify(row) for r, row in test.iterrows()]


print("Accuracy:", accuracy_score(y_true, y_pred))

print("\nClassification report:", classification_report(y_true, y_pred, labels=classes,zero_division=0))

print("\nConfusion matrix:", confusion_matrix(y_true, y_pred, labels=classes))

Accuracy: 0.6897880539499036

Classification report:               precision    recall  f1-score   support

       unacc       0.69      1.00      0.82       358
         acc       0.00      0.00      0.00       118
        good       0.00      0.00      0.00        19
       vgood       0.00      0.00      0.00        24

    accuracy                           0.69       519
   macro avg       0.17      0.25      0.20       519
weighted avg       0.48      0.69      0.56       519


Confusion matrix: [[358   0   0   0]
 [118   0   0   0]
 [ 19   0   0   0]
 [ 24   0   0   0]]
